## This is the training file for the local model which stands as a proof of concept currently

In [1]:
import os
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim import Adam

import sys, os
sys.path.append(os.path.expanduser("~/Desktop/projects/audio-filler"))

from models.model2.model2 import AudioEncoder1


In [15]:
class MusicGenreDataset(Dataset):
    def __init__(self, data_dir, clip_duration=15, stride=1, sample_rate=16000):
        self.data_dir = Path(os.path.expanduser(str(data_dir)))
        self.clip_length = clip_duration * sample_rate
        self.stride = stride * sample_rate
        self.sample_rate = sample_rate
        
        # Get genre labels
        self.genres = sorted([d.name for d in self.data_dir.iterdir() if d.is_dir()])
        self.genre_to_idx = {genre: idx for idx, genre in enumerate(self.genres)}
        
        # Collect audio files and their genres
        self.audio_files = []
        for genre in self.genres:
            genre_dir = self.data_dir / genre
            for audio_file in genre_dir.glob("*.mp3"):
                self.audio_files.append((audio_file, genre))
        
        # Precompute clip segments
        self.clips = []
        for audio_file, genre in self.audio_files:
            info = torchaudio.info(audio_file)
            total_samples = info.num_frames
            start = 0
            while start < total_samples:
                end = start + self.clip_length
                if end > total_samples:
                    if total_samples - start >= 10 * sample_rate:  # Check if at least 10s
                        end = total_samples
                        padding = self.clip_length - (end - start)
                        self.clips.append((audio_file, start, end, padding, genre))
                    break
                else:
                    self.clips.append((audio_file, start, end, 0, genre))
                start += self.stride

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        audio_file, start, end, padding, genre = self.clips[idx]
        waveform, sr = torchaudio.load(audio_file, frame_offset=start, num_frames=end-start)
        waveform = torchaudio.functional.resample(waveform, sr, self.sample_rate)[0]  # Convert to mono
        
        if padding > 0:
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        
        # Add channel dimension
        waveform = waveform.unsqueeze(0)
        label = self.genre_to_idx[genre]
        
        return waveform, label

In [16]:
# Initialize dataset and dataloader
dataset = MusicGenreDataset("~/data/project/music")
dataloader = DataLoader(dataset, batch_size=1000, shuffle=True, num_workers=4)

/var/folders/1g/bksky7bd2wb927qzgc2fqjhc0000gn/T/ipykernel_25220/2147094717.py:22: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  info = torchaudio.info(audio_file)
/opt/anaconda3/envs/audio-filler/lib/python3.11/site-packages/torchaudio/_backend/soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
/opt/anaconda3/envs/audio-filler/lib/python3.11/site-packages/tor

In [17]:
# length of batches
len(dataloader)

269

In [18]:
# Model and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioEncoder1().to(device)
optimizer = Adam(model.parameters(), lr=1e-4)

model.eval()

AudioEncoder1(
  (tanh): Tanh()
  (leaky_relu): LeakyReLU(negative_slope=0.2)
  (conv1): Conv1d(3, 32, kernel_size=(8,), stride=(4,), padding=(2,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(3,), padding=(1,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv1d(64, 128, kernel_size=(4,), stride=(2,))
  (encoder_transformer): TransformerBlock(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
    )
    (ff): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=128, bias=True)
    )
    (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (bn3): Ba

In [20]:
# Track metrics
metrics = {
    'total_loss': [], 'classification_loss': [], 
    'modulus_recon_loss': [], 'sign_recon_loss': [],
    'sign_accuracy': [], 'kl_loss': []
}

# Training loop
model.train()
for batch_idx, (data, targets) in enumerate(dataloader):
    data, targets = data.to(device), targets.to(device)
    
    optimizer.zero_grad()
    losses = model.loss_function(data, targets)
    losses['total_loss'].backward()
    optimizer.step()
    
    # Store metrics
    for k in metrics:
        metrics[k].append(losses[k].item())
    
    # Print batch statistics
    print(f"Batch {batch_idx+1}:")
    print(f"Total Loss: {losses['total_loss'].item():.4f}")
    print(f"Classification Loss: {losses['classification_loss'].item():.4f}")
    print(f"Modulus Recon Loss: {losses['modulus_recon_loss'].item():.4f}")
    print(f"Sign Recon Loss: {losses['sign_recon_loss'].item():.4f}")
    print(f"Sign Accuracy: {losses['sign_accuracy'].item():.4f}")
    print(f"KL Loss: {losses['kl_loss'].item():.4f}")
    
    # Plot every 5 batches
    if (batch_idx + 1) % 5 == 0:
        plt.figure(figsize=(12, 8))
        for i, (k, v) in enumerate(metrics.items()):
            plt.subplot(2, 3, i+1)
            plt.plot(v, label=k)
            plt.title(k)
            plt.xlabel('Batch')
        plt.tight_layout()
        plt.show()
        plt.close()

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/audio-filler/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/audio-filler/lib/python3.11/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/audio-filler/lib/python3.11/site-packages/torch/__init__.py", line 2264, in <module>
    from torch import quantization as quantization  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/audio-filler/lib/python3.11/site-packages/torch/quantization/__init__.py", line 2, in <module>
    from .fake_quantize import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/audio-filler/lib/python3.11/site-packages/torch/quantization/fake_quantize.py", 

KeyboardInterrupt: 

In [14]:
dataloader.dataset[0]

/opt/anaconda3/envs/audio-filler/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


(tensor([[ 0.0000,  0.0000,  0.0000,  ..., -0.0068, -0.0065, -0.0062]]), 0)